# LLM Zoomcamp 2026 – Homework 5: Monitoring

This notebook implements the Homework 5 tasks using OpenTelemetry and SQLite.

**Required files in the same folder:**
- `starter.py`
- `rag_helper.py`
- `.env` containing `OPENAI_API_KEY=...`


## 0. Install dependencies in the active notebook kernel

In [1]:
%pip install -q opentelemetry-api opentelemetry-sdk pandas python-dotenv openai minsearch gitsource
print("Dependencies installed. Restart the kernel once if imports still fail.")


[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
Dependencies installed. Restart the kernel once if imports still fail.


## 1. Check Python environment and files

In [2]:
import os
import sys
from pathlib import Path

print("Python:", sys.executable)
print("Working directory:", Path.cwd())
print("starter.py exists:", Path("starter.py").exists())
print("rag_helper.py exists:", Path("rag_helper.py").exists())
print(".env exists:", Path(".env").exists())

Python: /Users/veneraheddergott/LLM_zoomcamp/llm-zoomcamp_2026_vh-1/.venv/bin/python
Working directory: /Users/veneraheddergott/LLM_zoomcamp/llm-zoomcamp_2026_vh-1/05-monitoring/llm-zoomcamp-hw5
starter.py exists: True
rag_helper.py exists: True
.env exists: True


In [3]:
from dotenv import load_dotenv

load_dotenv()
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY is missing in .env"
print("OPENAI_API_KEY loaded successfully.")

OPENAI_API_KEY loaded successfully.


## 2. OpenTelemetry console exporter

In [4]:
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import ConsoleSpanExporter, SimpleSpanProcessor

# A provider can only be registered globally once per kernel.
# Restart the kernel before re-running this entire notebook from the top.
provider = TracerProvider()
provider.add_span_processor(SimpleSpanProcessor(ConsoleSpanExporter()))
trace.set_tracer_provider(provider)
tracer = trace.get_tracer("llm-zoomcamp")

print("OpenTelemetry console exporter configured.")

OpenTelemetry console exporter configured.


## 3. Load the starter RAG

In [5]:
from pathlib import Path

print("Aktueller Ordner:", Path.cwd())
print("Dateien:", [p.name for p in Path.cwd().iterdir()])

Aktueller Ordner: /Users/veneraheddergott/LLM_zoomcamp/llm-zoomcamp_2026_vh-1/05-monitoring/llm-zoomcamp-hw5
Dateien: ['uv.lock', 'pyproject.toml', '__pycache__', 'traces.db', 'README.md', 'rag_helper.py', '.env', '.venv', '.python-version', '.ipynb_checkpoints', 'homework5.ipynb', 'main.py', 'starter.py']


In [6]:
import inspect
import starter

rag = starter.rag
RAGBase = type(rag)

print("RAG class:", RAGBase)
print("RAG attributes:", sorted(vars(rag).keys()))
print("rag() signature:", inspect.signature(rag.rag))
print("search() signature:", inspect.signature(rag.search))
print("llm() signature:", inspect.signature(rag.llm))

RAG class: <class 'rag_helper.RAGBase'>
RAG attributes: ['index', 'instructions', 'llm_client', 'model', 'prompt_template']
rag() signature: (query)
search() signature: (query, num_results=5)
llm() signature: (prompt)


## Q1. First trace

Create one span for each of these methods:
- `rag`
- `search`
- `llm`

**Question:** How many spans does the trace produce?


In [7]:
class RAGTraced(RAGBase):
    def search(self, *args, **kwargs):
        with tracer.start_as_current_span("search"):
            return super().search(*args, **kwargs)

    def llm(self, *args, **kwargs):
        with tracer.start_as_current_span("llm"):
            return super().llm(*args, **kwargs)

    def rag(self, *args, **kwargs):
        with tracer.start_as_current_span("rag"):
            return super().rag(*args, **kwargs)


# Reuse the already initialized starter object without guessing constructor arguments.
traced_rag = object.__new__(RAGTraced)
traced_rag.__dict__.update(rag.__dict__)

print("RAGTraced created successfully.")

RAGTraced created successfully.


In [8]:
query = "How does the agentic loop keep calling the model until it stops?"

answer = traced_rag.rag(query)
print(answer)


{
    "name": "search",
    "context": {
        "trace_id": "0x91c4a3f17a20f90f7efc0152634ea64d",
        "span_id": "0xa1b9400cdca035d5",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0x73f52eda28c141e3",
    "start_time": "2026-07-26T15:53:03.957755Z",
    "end_time": "2026-07-26T15:53:03.962215Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {},
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.44.0",
            "service.instance.id": "7a7eb039-9c95-4eac-8f2d-e721699683d7",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}
{
    "name": "llm",
    "context": {
        "trace_id": "0x91c4a3f17a20f90f7efc0152634ea64d",
        "span_id": "0xdfc1482a86c28480",
        "trace_state": "[]"
    },
    "kind": "SpanKind

## Q2. Capture token usage and cost

**Question: How many input tokens do we see for the LLM call?**

In [9]:
# Adjust these prices only if your selected model uses different rates.
# Values are USD per 1,000,000 tokens.
INPUT_PRICE_PER_MILLION = 0.0
OUTPUT_PRICE_PER_MILLION = 0.0


class RAGTracedWithMetrics(RAGBase):
    def search(self, *args, **kwargs):
        with tracer.start_as_current_span("search"):
            return super().search(*args, **kwargs)

    def llm(self, *args, **kwargs):
        with tracer.start_as_current_span("llm") as span:
            response = super().llm(*args, **kwargs)

            usage = getattr(response, "usage", None)
            if usage is not None:
                input_tokens = getattr(usage, "input_tokens", None)
                output_tokens = getattr(usage, "output_tokens", None)

                if input_tokens is not None:
                    span.set_attribute("input_tokens", input_tokens)
                if output_tokens is not None:
                    span.set_attribute("output_tokens", output_tokens)

                if input_tokens is not None and output_tokens is not None:
                    cost = (
                        input_tokens * INPUT_PRICE_PER_MILLION
                        + output_tokens * OUTPUT_PRICE_PER_MILLION
                    ) / 1_000_000
                    span.set_attribute("cost", cost)

            return response

    def rag(self, *args, **kwargs):
        with tracer.start_as_current_span("rag"):
            return super().rag(*args, **kwargs)


traced_rag_metrics = object.__new__(RAGTracedWithMetrics)
traced_rag_metrics.__dict__.update(rag.__dict__)

print("Metric-enabled RAG created.")

Metric-enabled RAG created.


In [10]:
answer = traced_rag_metrics.rag(query)
print(answer)

print("\nQ2: Read input_tokens from the llm span output and select the closest option.")

{
    "name": "search",
    "context": {
        "trace_id": "0xbcd4df8e481ec2489ed3260021f03013",
        "span_id": "0x460b1af9ae806c8d",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0x0571850fc3e6d36b",
    "start_time": "2026-07-26T15:53:16.549528Z",
    "end_time": "2026-07-26T15:53:16.552729Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {},
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.44.0",
            "service.instance.id": "7a7eb039-9c95-4eac-8f2d-e721699683d7",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}
{
    "name": "llm",
    "context": {
        "trace_id": "0xbcd4df8e481ec2489ed3260021f03013",
        "span_id": "0x74a7ddcde0fd103d",
        "trace_state": "[]"
    },
    "kind": "SpanKind

## Q3. Span timing

In the console output, compare `start_time` and `end_time` for the `llm` span. The later SQLite analysis will calculate durations automatically.

**Question: For a typical query, roughly how long does the LLM call take?**

In [11]:
from datetime import datetime

llm_start = "2026-07-26T15:15:40.206256Z"
llm_end = "2026-07-26T15:15:42.242619Z"

start_time = datetime.fromisoformat(llm_start.replace("Z", "+00:00"))
end_time = datetime.fromisoformat(llm_end.replace("Z", "+00:00"))

duration_ms = (end_time - start_time).total_seconds() * 1000

print(f"LLM duration: {duration_ms:.2f} ms")

if duration_ms < 100:
    answer = "Under 100ms"
elif duration_ms < 500:
    answer = "100-500ms"
elif duration_ms <= 2000:
    answer = "500-2000ms"
else:
    answer = "Over 2000ms"

print(f"Q3 answer: {answer}")

LLM duration: 2036.36 ms
Q3 answer: Over 2000ms


## Q4. Save spans to SQLite

**Question: Which span names appear in the spans table?**

In [12]:
from pathlib import Path

db_path = Path("traces.db")

if db_path.exists():
    db_path.unlink()
    print("Old traces.db deleted.")
else:
    print("No old traces.db found.")

Old traces.db deleted.


In [13]:
import sqlite3

from opentelemetry.sdk.trace.export import (
    SpanExporter,
    SpanExportResult,
)


class SQLiteSpanExporter(SpanExporter):

    def __init__(self, db_path="traces.db"):
        self.db_path = db_path

        with sqlite3.connect(self.db_path) as conn:
            conn.execute("""
                CREATE TABLE IF NOT EXISTS spans (
                    name TEXT,
                    start_time INTEGER,
                    end_time INTEGER,
                    input_tokens INTEGER,
                    output_tokens INTEGER,
                    cost REAL
                )
            """)

    def export(self, spans):
        with sqlite3.connect(self.db_path) as conn:
            for span in spans:
                attrs = dict(span.attributes or {})

                conn.execute(
                    """
                    INSERT INTO spans (
                        name,
                        start_time,
                        end_time,
                        input_tokens,
                        output_tokens,
                        cost
                    )
                    VALUES (?, ?, ?, ?, ?, ?)
                    """,
                    (
                        span.name,
                        span.start_time,
                        span.end_time,
                        attrs.get("input_tokens"),
                        attrs.get("output_tokens"),
                        attrs.get("cost"),
                    ),
                )

            conn.commit()

        return SpanExportResult.SUCCESS

    def shutdown(self):
        pass

    def force_flush(self, timeout_millis=30000):
        return True


print("SQLiteSpanExporter defined.")

SQLiteSpanExporter defined.


### Important

Restart the kernel now, then run the next setup cell instead of the earlier console-provider cell. OpenTelemetry allows only one global provider per kernel.

In [14]:
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import SimpleSpanProcessor


sqlite_provider = TracerProvider()

sqlite_provider.add_span_processor(
    SimpleSpanProcessor(
        SQLiteSpanExporter("traces.db")
    )
)

tracer = sqlite_provider.get_tracer("llm-zoomcamp")

print("SQLite tracer configured.")

SQLite tracer configured.


In [15]:
tracer = sqlite_provider.get_tracer("llm-zoomcamp")

In [16]:
from starter import rag


query = "How does the agentic loop keep calling the model until it stops?"

print("RAG loaded.")
print(query)

RAG loaded.
How does the agentic loop keep calling the model until it stops?


In [17]:
class RAGTracedWithMetrics(rag.__class__):

    def search(self, query):
        with tracer.start_as_current_span("search"):
            return super().search(query)

    def llm(self, messages):
        with tracer.start_as_current_span("llm") as span:
            response = super().llm(messages)

            usage = response.usage

            span.set_attribute(
                "input_tokens",
                usage.input_tokens
            )

            span.set_attribute(
                "output_tokens",
                usage.output_tokens
            )

            span.set_attribute(
                "cost",
                0.0
            )

            return response

    def rag(self, query):
        with tracer.start_as_current_span("rag"):
            return super().rag(query)


print("RAGTracedWithMetrics defined.")

RAGTracedWithMetrics defined.


In [18]:
traced_rag_metrics = object.__new__(RAGTracedWithMetrics)

traced_rag_metrics.__dict__.update(
    rag.__dict__
)

print("traced_rag_metrics created.")

traced_rag_metrics created.


In [19]:
answer = traced_rag_metrics.rag(query)

print(answer)

It keeps calling the model in a `while True` loop.

After each model response, the code checks whether there were any `function_call` items:

- if there were function calls, it runs the tools, appends the tool outputs to `messages`, and loops again
- if there were no function calls, it breaks

So the stop condition is:

```python
if has_function_calls == False:
    break
```

In other words, the agent keeps re-calling the model until the model returns a final message with no more tool calls.


In [20]:
with sqlite3.connect("traces.db") as conn:
    rows = conn.execute(
        """
        SELECT name
        FROM spans
        ORDER BY start_time
        """
    ).fetchall()

print(rows)

[('rag',), ('search',), ('llm',)]


In [21]:
import pandas as pd


with sqlite3.connect("traces.db") as conn:
    q4_results = pd.read_sql_query(
        """
        SELECT DISTINCT name
        FROM spans
        ORDER BY name
        """,
        conn
    )

q4_results

,name
0,llm
1,rag
2,search


## Q5. Which child span takes the most total time?

**Question: Excluding the rag span, which span type takes the most total time?**

In [22]:
import sqlite3
import pandas as pd

with sqlite3.connect("traces.db") as conn:
    spans_df = pd.read_sql_query(
        """
        SELECT
            name,
            start_time,
            end_time
        FROM spans
        """,
        conn
    )

display(spans_df)

,name,start_time,end_time
0,search,1785081198068775000,1785081198071070000
1,llm,1785081198119462000,1785081200018805000
2,rag,1785081198068730000,1785081200026240000


In [23]:
spans_df["duration_ms"] = (
    spans_df["end_time"] - spans_df["start_time"]
) / 1_000_000

duration_summary = (
    spans_df.loc[spans_df["name"] != "rag"]
    .groupby("name", as_index=False)["duration_ms"]
    .sum()
    .sort_values("duration_ms", ascending=False)
)

display(duration_summary)

if not duration_summary.empty:
    print("Q5 answer:", duration_summary.iloc[0]["name"])

,name,duration_ms
0,llm,1899.343
1,search,2.295


Q5 answer: llm


## Q6. Token stability across four runs

**Question: How much do the input tokens vary across 4 runs of the same query?**

In [24]:
# The database already contains at least one call from Q4.
# Run the same query three additional times.
for run_number in range(1, 4):
    print(f"Run {run_number}/3")
    traced_rag_metrics.rag(query)

print("Three additional calls completed.")

Run 1/3
Run 2/3
Run 3/3
Three additional calls completed.


In [25]:
with sqlite3.connect("traces.db") as conn:
    llm_tokens = pd.read_sql_query(
        """
        SELECT rowid, input_tokens
        FROM spans
        WHERE name = 'llm' AND input_tokens IS NOT NULL
        ORDER BY rowid DESC
        LIMIT 4
        """,
        conn,
    ).sort_values("rowid")

display(llm_tokens)

tokens = llm_tokens["input_tokens"].astype(float)

if len(tokens) == 4:
    minimum = tokens.min()
    maximum = tokens.max()
    variation = 0.0 if minimum == 0 else (maximum - minimum) / minimum

    print(f"Minimum: {minimum:.0f}")
    print(f"Maximum: {maximum:.0f}")
    print(f"Variation: {variation:.2%}")

    if variation == 0:
        answer_q6 = "They're identical"
    elif variation <= 0.10:
        answer_q6 = "Within 10% of each other"
    elif variation <= 0.50:
        answer_q6 = "Within 50% of each other"
    else:
        answer_q6 = "They vary more than 50%"

    print("Q6 answer:", answer_q6)
else:
    print("Expected 4 llm rows, but found:", len(tokens))

,rowid,input_tokens
3,2,7111
2,5,7111
1,8,7111
0,11,7111


Minimum: 7111
Maximum: 7111
Variation: 0.00%
Q6 answer: They're identical
